# `sim.bus`: the fake I2C bus and generic parts
**Host only.** The simulation never runs on the board, so this notebook has no hardware
section. Every cell runs in ordinary Python (CPython) on this computer.

# TODO:
Nothing open.

Decided (2026-11-16): only what is not one particular part lives here: the bus, a generic
register file, a generic one-byte mux and a canned-reply command part. Each real part's fake
(`FakeSHTC3`, `FakeTMP102`, ...) is written in that part's device notebook, next to its
driver, as `devices/<type>/.../<part>_fake.py`.

# Python file
location: `src/micropython_i2c/sim/bus.py`

tests: `tests/test_sim_bus.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/sim/bus.py
"""``FakeBus``, a ``machine.I2C`` stand-in with mux routing, and generic fake parts.

``FakeBus`` models mux routing: a part attached behind ``[(mux_addr, ch), ...]`` answers only
while every mux on that path has the channel enabled, the way real silicon does. A part is any
object with some of ``raw_write(data)``, ``raw_read(n)``, ``mem_write(reg, data)``,
``mem_read(reg, n)``, and optionally ``hear(byte)`` (sees every address byte on the bus) and
``bus_addr`` (the address it answers at, if it can change). A part refuses a transfer the way a
real one NACKs: by raising ``OSError``.

Generic parts: ``FakeRegisterPart`` (an 8-bit register file), ``FakeMux`` (one control byte, one
bit per channel) and ``FakeSensirion`` (canned 16-bit words, each followed by a CRC). Each real
part's fake sits next to its driver as ``<part>_fake.py``.

Host only.

Generated by ``dev_notebooks/sim_bus_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None.

## Internal
None: the bus only moves bytes.

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_sim_bus.py
import pytest

from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart, FakeSensirion

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c import I2CConnection

# Intent
**Purpose:** let the drivers, the connection layer and the manager run on the host exactly as
on the board, with fake parts at the other end of the wire.

**Why it exists:** a driver only ever calls the `machine.I2C` methods (`writeto`, `readfrom`,
`readfrom_mem`, ...). `FakeBus` has the same methods and hands each transfer to whichever fake
part answers at that address, so a driver cannot tell it is not on the board. Mux routing is
the part that makes this worth having: a mistake in path selection shows up as `ENODEV`, the
same as on the bench.

```mermaid
flowchart LR
    D[driver] --> C[I2CConnection] --> B[FakeBus]
    B -->|0x70 always visible| M[FakeMux]
    B -->|0x48 only while mux ch 2 on| T[FakeTMP102]
    B -->|0x18| R[FakeRelay]
```

| Event | What `FakeBus` does |
|---|---|
| write to an address | every visible part there gets it (a write cannot collide) |
| read from an address | the one visible part there answers; two is a `RuntimeError` (a collision) |
| nothing visible at the address | `OSError(19, "ENODEV")`, as `machine.I2C` raises for a NACK |
| any transfer | every visible part with `hear` sees the address byte first |

**Not its job:** electrical faults (pull-ups, clock speed, a stuck line) and timing on the wire.

## What the module defines

| Name | What it is |
|---|---|
| `FakeBus` | the `machine.I2C` stand-in; `attach(addr, part, path)`; `log` of every transfer |
| `FakeRegisterPart` | 8-bit register file with auto-increment and read-only bits |
| `FakeMux` | one control byte, one bit per channel (the TCA9548A scheme) |
| `FakeSensirion` | a command part returning canned words, each followed by a CRC |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/sim/bus.py -a
__all__ = ["FakeBus", "FakeMux", "FakeRegisterPart", "FakeSensirion"]

## The generic parts
Three small parts the base-layer tests use, and the device notebooks build on. They come
before `FakeBus` so its validation can use them.

## `FakeRegisterPart`
An 8-bit register file. A read or write starting at `reg` continues at `reg + 1` (auto-
increment, wrapping at `size`). `read_only` maps a register to the mask of bits that keep their
value on a write. A bare `raw_read` reads from register 0; `raw_write` is ignored.

In [ ]:
%%extract ../src/micropython_i2c/sim/bus.py -a
class FakeRegisterPart:
    """A register-mapped part with an auto-incrementing 8-bit register file.

    Args:
        regs: ``{register: value}`` to start with; everything else is 0.
        read_only: ``{register: mask}``: bits set in ``mask`` keep their value on a write.
        size: Number of registers.

    Attributes:
        regs: The register file (``bytearray``).
    """

    def __init__(self, regs=None, read_only=None, size=256):
        self.regs = bytearray(size)
        for reg, val in (regs or {}).items():
            self.regs[reg] = val
        self.read_only = read_only or {}

    def mem_read(self, reg, nbytes):
        return bytes(self.regs[(reg + i) % len(self.regs)] for i in range(nbytes))

    def mem_write(self, reg, data):
        for i, b in enumerate(data):
            r = (reg + i) % len(self.regs)
            keep = self.read_only.get(r, 0)
            self.regs[r] = (self.regs[r] & keep) | (b & ~keep & 0xFF)

    def raw_read(self, nbytes):
        return self.mem_read(0, nbytes)

    def raw_write(self, data):
        pass

## `FakeMux`
One control byte: bit `n` enables channel `n`, and a write replaces the whole byte (the
TCA9548A scheme). A read returns the byte. The TCA9548A's own fake, with its address range and
reset, builds on this in `muxes/tca9548a_devNB`.

In [ ]:
%%extract ../src/micropython_i2c/sim/bus.py -a
class FakeMux:
    """A mux with one control byte, one bit per channel.

    Attributes:
        state: The control byte: bit ``n`` set means channel ``n`` is connected.
    """

    def __init__(self):
        self.state = 0

    def raw_write(self, data):
        self.state = data[-1]

    def raw_read(self, nbytes):
        return bytes([self.state] * nbytes)

## `FakeSensirion`
A command part that answers every read with the same canned 16-bit words, each followed by
`crc_fn` of its two bytes (Sensirion parts send a CRC-8 after every word). Every command
written is logged in `commands`, so a test can check what the driver sent.

In [ ]:
%%extract ../src/micropython_i2c/sim/bus.py -a
class FakeSensirion:
    """A command part returning canned words, each followed by ``crc_fn(word bytes)``.

    Args:
        words: The 16-bit words every read returns, in order.
        crc_fn: ``crc_fn(two bytes) -> int``, the checksum sent after each word.

    Attributes:
        commands: Every command written, as an int.
    """

    def __init__(self, words, crc_fn):
        self.words = words
        self.crc_fn = crc_fn
        self.commands = []

    def raw_write(self, data):
        self.commands.append(int.from_bytes(data, "big"))

    def raw_read(self, nbytes):
        out = b""
        for w in self.words:
            pair = w.to_bytes(2, "big")
            out += pair + bytes([self.crc_fn(pair)])
        return out[:nbytes]

## `FakeBus`
The bus. `attach(addr, part, path)` puts a part at `addr` behind `path`
(`[(mux_addr, channel), ...]`, outermost first; empty for the root segment). A part is visible
when every mux on its path is itself visible and has that channel on. A part with a
`bus_addr` attribute answers there instead of at the address it was attached at (the VL53L1X
and TMAG5273 can change their address).

A part with a `hear(byte)` method gets every address byte on its segment before the transfer
(the TMAG5273's CRC covers address bytes it overheard). Visibility is checked before the
transfer: a mux switches only after its own address byte.

In [ ]:
%%extract ../src/micropython_i2c/sim/bus.py -a
class FakeBus:
    """Stands in for ``machine.I2C``. Attach parts with ``attach(addr, part, path)``.

    Attributes:
        nodes: ``[(path tuple, addr, part), ...]`` in attach order.
        log: Every transfer: ``("w", addr, data)``, ``("r", addr, n)``,
            ``("wm", addr, reg, data)``, ``("rm", addr, reg, n)``.
    """

    def __init__(self):
        self.nodes = []
        self.log = []

    def attach(self, addr, part, path=()):
        """Put ``part`` at ``addr`` behind ``path``; returns ``part``."""
        self.nodes.append((tuple(tuple(h) for h in path), addr, part))
        return part

    def _mux_at(self, path, addr):
        for p, a, part in self.nodes:
            if p == path and a == addr:
                return part
        raise LookupError("no mux at {:#04x} on {}".format(addr, path))

    def _visible(self, path):
        for i, (maddr, ch) in enumerate(path):
            if not self._visible(path[:i]):
                return False
            if not self._mux_at(path[:i], maddr).state & (1 << ch):
                return False
        return True

    @staticmethod
    def _addr_of(addr, part):
        """The address a part answers at: its ``bus_addr`` if it has one, else ``addr``."""
        return getattr(part, "bus_addr", addr)

    def _targets(self, addr):
        hits = [
            part for p, a, part in self.nodes if self._addr_of(a, part) == addr and self._visible(p)
        ]
        if not hits:
            raise OSError(19, "ENODEV")
        return hits

    def _target(self, addr):
        """The one part a read talks to. Two parts at one address answering a read is a
        collision (their data would be mixed); a write simply reaches both."""
        hits = self._targets(addr)
        if len(hits) > 1:
            raise RuntimeError("bus collision at {:#04x}".format(addr))
        return hits[0]

    def scan(self):
        return sorted({self._addr_of(a, part) for p, a, part in self.nodes if self._visible(p)})

    def _hear(self, addr, read):
        """Give the address byte to every visible part that has a ``hear`` hook."""
        byte = (addr << 1) | (1 if read else 0)
        for path, _, part in self.nodes:
            if hasattr(part, "hear") and self._visible(path):
                part.hear(byte)

    def writeto(self, addr, data, stop=True):
        self.log.append(("w", addr, bytes(data)))
        self._hear(addr, False)
        for part in self._targets(addr):  # every part at the address receives a write
            part.raw_write(bytes(data))
        return len(data)

    def readfrom(self, addr, nbytes, stop=True):
        self.log.append(("r", addr, nbytes))
        self._hear(addr, True)
        return self._target(addr).raw_read(nbytes)

    def readfrom_into(self, addr, buf, stop=True):
        buf[:] = self.readfrom(addr, len(buf))

    def writeto_mem(self, addr, reg, data, addrsize=8):
        self.log.append(("wm", addr, reg, bytes(data)))
        self._hear(addr, False)
        self._target(addr).mem_write(reg, bytes(data))

    def readfrom_mem(self, addr, reg, nbytes, addrsize=8):
        self.log.append(("rm", addr, reg, nbytes))
        self._hear(addr, False)
        self._hear(addr, True)  # the repeated start
        return self._target(addr).mem_read(reg, nbytes)

    def readfrom_mem_into(self, addr, reg, buf, addrsize=8):
        buf[:] = self.readfrom_mem(addr, reg, len(buf), addrsize)

### validation/test
A demo part for the cells below: it records every address byte it hears.

In [ ]:
%%extract ../tests/test_sim_bus.py -a
class Listener(FakeRegisterPart):
    """A register part that records the address bytes it hears."""

    def __init__(self):
        super().__init__()
        self.heard = []

    def hear(self, byte):
        self.heard.append(byte)

#### validation: nominal usage
A mux at `0x70` with a register part at `0x48` on channel 2. The part is invisible until the
mux switches channel 2 on.

In [ ]:
bus = FakeBus()
mux = bus.attach(0x70, FakeMux())
part = bus.attach(0x48, FakeRegisterPart({0: 0x19}), path=[(0x70, 2)])
print("channel off, scan:", [hex(a) for a in bus.scan()])
bus.writeto(0x70, bytes([1 << 2]))
print("channel 2 on, scan:", [hex(a) for a in bus.scan()], "| reg 0:", bus.readfrom_mem(0x48, 0, 1))
print("log:", bus.log)

The connection layer on top: `I2CConnection` with a path selects the channel itself on every
transfer.

In [ ]:
bus2 = FakeBus()
bus2.attach(0x70, FakeMux())
bus2.attach(0x48, FakeRegisterPart({0: 0x19, 1: 0x00}), path=[(0x70, 2)])
conn = I2CConnection(bus2, 0x48, path=[[0x70, 2]])
print("read through the connection:", conn.readfrom_mem(0, 2))

#### validation: anti-nominal usage

In [ ]:
bus = FakeBus()
bus.attach(0x70, FakeMux())
bus.attach(0x48, FakeRegisterPart(), path=[(0x70, 2)])
try:
    bus.readfrom_mem(0x48, 0, 1)  # the channel is off: nothing answers at 0x48
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a read behind a closed channel should have raised OSError")

In [ ]:
bus = FakeBus()
bus.attach(0x48, FakeRegisterPart())
bus.attach(0x48, FakeRegisterPart())
try:
    bus.readfrom(0x48, 1)  # two parts answer at one address
except RuntimeError as err:
    print("raised RuntimeError, as expected:\n    {}".format(err))
else:
    raise AssertionError("two parts at one address should have raised RuntimeError on a read")

#### unit

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_part_behind_a_mux_answers_only_while_its_channel_is_on():
    bus = FakeBus()
    bus.attach(0x70, FakeMux())
    bus.attach(0x48, FakeRegisterPart({0: 0x42}), path=[(0x70, 2)])
    with pytest.raises(OSError):
        bus.readfrom_mem(0x48, 0, 1)
    bus.writeto(0x70, bytes([0b100]))
    assert bus.readfrom_mem(0x48, 0, 1) == b"\x42"
    bus.writeto(0x70, bytes([0b001]))
    assert bus.scan() == [0x70]

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_nested_muxes_need_every_hop_on():
    bus = FakeBus()
    bus.attach(0x70, FakeMux())
    bus.attach(0x71, FakeMux(), path=[(0x70, 1)])
    bus.attach(0x48, FakeRegisterPart(), path=[(0x70, 1), (0x71, 3)])
    bus.writeto(0x70, bytes([0b10]))
    bus.writeto(0x71, bytes([0b1000]))
    assert 0x48 in bus.scan()
    bus.writeto(0x70, bytes([0]))
    assert bus.scan() == [0x70]

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_two_parts_at_one_address_collide_on_read_but_both_get_a_write():
    bus = FakeBus()
    a = bus.attach(0x48, FakeSensirion([1], lambda b: 0))
    b = bus.attach(0x48, FakeSensirion([2], lambda b: 0))
    bus.writeto(0x48, b"\x12\x34")
    assert a.commands == b.commands == [0x1234]
    with pytest.raises(RuntimeError, match="collision at 0x48"):
        bus.readfrom(0x48, 3)

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_a_part_with_bus_addr_answers_there():
    bus = FakeBus()
    part = bus.attach(0x29, FakeRegisterPart())
    part.bus_addr = 0x30
    assert bus.scan() == [0x30]
    with pytest.raises(OSError):
        bus.readfrom(0x29, 1)
    bus.readfrom(0x30, 1)

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_visible_parts_hear_every_address_byte_first():
    bus = FakeBus()
    ear = bus.attach(0x22, Listener())
    bus.attach(0x48, FakeRegisterPart())
    bus.readfrom_mem(0x48, 0, 1)
    bus.writeto(0x48, b"\x00")
    assert ear.heard == [0x90, 0x91, 0x90]  # write + repeated-start read, then a write

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_register_part_auto_increments_and_keeps_read_only_bits():
    part = FakeRegisterPart({0xFF: 0xAA, 0: 0xBB}, read_only={1: 0xF0}, size=256)
    assert part.mem_read(0xFF, 2) == b"\xaa\xbb"  # wraps at size
    part.mem_write(1, b"\xff")
    assert part.regs[1] == 0x0F

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_sensirion_part_sends_a_crc_after_each_word():
    part = FakeSensirion([0xBEEF, 0x0001], lambda pair: pair[0] ^ pair[1])
    assert part.raw_read(6) == bytes([0xBE, 0xEF, 0xBE ^ 0xEF, 0x00, 0x01, 0x01])
    assert part.raw_read(2) == b"\xbe\xef"

In [ ]:
%%extract ../tests/test_sim_bus.py -a --strip-ipytest
%%ipytest


def test_log_records_every_transfer():
    bus = FakeBus()
    bus.attach(0x48, FakeRegisterPart())
    bus.writeto_mem(0x48, 1, b"\x60")
    bus.readfrom_mem(0x48, 1, 1)
    bus.writeto(0x48, b"\x00")
    bus.readfrom(0x48, 2)
    assert bus.log == [
        ("wm", 0x48, 1, b"\x60"),
        ("rm", 0x48, 1, 1),
        ("w", 0x48, b"\x00"),
        ("r", 0x48, 2),
    ]